# 4.5 · Lasso 回归 / Lasso Regression (L1)

> **课程定位 / Where this fits**
> 第 5 课，**Part 4 · 监督学习：回归**。
> Lesson 5, **Part 4 · Supervised Regression**.
>
> 4.4 的 Ridge 用 L2 惩罚，系数只收缩不归零。**Lasso** 把惩罚换成 **L1**（系数绝对值之和），会把**部分系数精确压到 0**——等于自动做了**特征选择**。"为什么 L1 能产生稀疏解而 L2 不能"是 ML 面试最经典的几何题之一。
> Ridge (4.4) used L2 and only shrinks. **Lasso** swaps in **L1** (sum of absolute coefficients), which drives **some coefficients exactly to 0** — automatic **feature selection**. "Why does L1 give sparsity but L2 doesn't" is a classic ML interview geometry question.
>
> 💼 **实战/面试视角**："L1 vs L2 / 为什么 Lasso 稀疏 / 什么时候用 Lasso" 是**必考**。
> 💼 **Practical/interview angle:** "L1 vs L2 / why Lasso is sparse / when to use Lasso" are must-asks.

> 📐 **符号约定 / Notation**
> - L1 惩罚 $=\lambda\sum_j |w_j| = \lambda\|\mathbf{w}\|_1$ —— 系数绝对值之和
> - 稀疏 sparse —— 大部分系数为 0 / most coefficients are 0

> 💡 **面试相关 / Interview-relevant**
> - "L1 vs L2 正则的区别"（出镜率 ★★★★★）
> - "为什么 L1 产生稀疏解（几何/菱形解释）"（出镜率 ★★★★★）
> - "Lasso 怎么做特征选择"（★★★★★）
> - "Lasso vs Ridge 何时用哪个"（★★★★）
> - "软阈值算子是什么"（★★★）

---

## 学习目标 / Learning Objectives

1. 用**几何（圆 vs 菱形）**解释为什么 L1 稀疏、L2 不稀疏。
   Explain via geometry (circle vs diamond) why L1 is sparse and L2 isn't.
2. 理解**软阈值算子**——L1 稀疏的算子机制。
   Understand the soft-thresholding operator — the mechanism of L1 sparsity.
3. 看 **Lasso 路径**如何逐个把系数归 0。
   Watch the Lasso path zero out coefficients one by one.
4. 用 Lasso 做**特征选择**（剔除噪声特征）。
   Use Lasso for feature selection (drop noise features).
5. 知道 **Lasso vs Ridge** 何时用哪个。
   Know when to use Lasso vs Ridge.

## 目录 / TOC
1. [先建直觉：圆 vs 菱形 ⭐](#1)
2. [🏠 数据：真实特征 + 噪声特征](#2)
3. [软阈值：稀疏的算子机制 ⭐](#3)
4. [Lasso 路径：逐个归 0 ⭐](#4)
5. [特征选择 + vs Ridge ⭐](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 先建直觉：圆 vs 菱形 ⭐ / Intuition: Circle vs Diamond

为什么 L1 稀疏、L2 不稀疏？最经典的解释是**几何**。正则化等价于"在一个'预算约束'内，找让损失最小的系数"。损失的等高线是**椭圆**，约束区域的形状决定了最优解长什么样：
Why is L1 sparse but L2 not? The classic explanation is **geometric**. Regularization is equivalent to "minimize loss within a 'budget' constraint". The loss contours are **ellipses**; the shape of the constraint region determines what the solution looks like:
- **L2（Ridge）的约束是个圆**：椭圆通常碰到圆的**光滑曲边**——接触点两个坐标都非零。
  **L2's constraint is a circle**: the ellipse touches its **smooth curved edge** — both coordinates nonzero.
- **L1（Lasso）的约束是个菱形**：菱形有**尖角，且尖角正好在坐标轴上**。椭圆很容易先碰到尖角——接触点有一个坐标 = 0（稀疏！）。
  **L1's constraint is a diamond**: the diamond has **corners, and they sit exactly on the axes**. The ellipse easily hits a corner first — where one coordinate = 0 (sparse!).

维度越高，菱形的"尖角/边"越多落在坐标轴/坐标平面上，所以高维下 Lasso 会产生大量 0。这就是 L1 稀疏的几何本质。
In higher dimensions, more of the diamond's corners/edges lie on axes, so Lasso produces many zeros. That's the geometric essence of L1 sparsity.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
w1 = np.linspace(-2, 3, 400); w2 = np.linspace(-2, 3, 400)
W1, W2 = np.meshgrid(w1, w2)
center = np.array([1.8, 1.2])              # 假设 OLS 最优解在这里
A = np.array([[2, 0.8], [0.8, 1]])         # 损失的二次型(决定椭圆形状)
Loss = A[0,0]*(W1-center[0])**2 + 2*A[0,1]*(W1-center[0])*(W2-center[1]) + A[1,1]*(W2-center[1])**2

for ax, norm in [(axes[0], "L2 (Ridge): 约束=圆 circle"), (axes[1], "L1 (Lasso): 约束=菱形 diamond")]:
    ax.contour(W1, W2, Loss, levels=12, cmap="Blues", alpha=0.6)   # 损失等高线(椭圆)
    theta = np.linspace(0, 2*np.pi, 200)
    if "L2" in norm:
        ax.plot(1.2*np.cos(theta), 1.2*np.sin(theta), "r-", lw=2)    # 圆形约束
        ax.scatter([1.05], [0.55], c="red", s=120, zorder=5, label="解(两坐标都非零)")
    else:
        ax.plot([1.2,0,-1.2,0,1.2], [0,1.2,0,-1.2,0], "r-", lw=2)    # 菱形约束(尖角在轴上)
        ax.scatter([0], [1.2], c="red", s=150, marker="*", zorder=5, label="解(w1=0, 稀疏!)")
    ax.scatter(*center, c="green", s=80, zorder=5, label="OLS 解")
    ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
    ax.set_xlabel("w1"); ax.set_ylabel("w2"); ax.set_title(norm); ax.legend(fontsize=8); ax.set_aspect("equal")
plt.tight_layout(); plt.show()
print("L2: 椭圆碰圆的光滑边 → w1,w2 都非零")
print("L1: 椭圆碰菱形尖角(在 w2 轴上) → w1=0, 稀疏! 这就是 Lasso 特征选择的几何原理")


<a id="2"></a>
## 2. 数据：真实特征 + 噪声特征 / Data: Real + Noise Features

为了展示 Lasso 的特征选择能力，我们给 California Housing 的 8 个真实特征**人为加上 20 个纯噪声特征**，看 Lasso 能不能自动把噪声筛掉。
To showcase Lasso's feature selection, we add **20 pure-noise features** to California Housing's 8 real ones and see if Lasso filters the noise out automatically.


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

data = fetch_california_housing(as_frame=True)
X, y = data.data.values, data.target.values
noise = rng.normal(size=(len(X), 20))           # 20 列与目标无关的纯噪声 / pure noise columns
X_aug = np.c_[X, noise]
feat_names = list(data.feature_names) + [f"noise_{i}" for i in range(20)]

X_tr, X_te, y_tr, y_te = train_test_split(X_aug, y, test_size=0.3, random_state=0)
scaler = StandardScaler().fit(X_tr)              # Lasso 也必须标准化(同 Ridge)
Xtr, Xte = scaler.transform(X_tr), scaler.transform(X_te)
print(f"8 真实特征 + 20 噪声特征 = {X_aug.shape[1]} 维; 看 Lasso 能否筛掉噪声")


<a id="3"></a>
## 3. 软阈值：稀疏的算子机制 ⭐ / Soft-Thresholding

几何解释了"为什么"，**软阈值算子**解释了"怎么实现"。Lasso 求解时，每个系数的更新都经过软阈值函数 $S_\lambda(\rho)=\text{sign}(\rho)\max(|\rho|-\lambda, 0)$：
Geometry explains "why"; the **soft-thresholding operator** explains "how". In solving Lasso, each coefficient passes through $S_\lambda(\rho)=\text{sign}(\rho)\max(|\rho|-\lambda, 0)$:
- 当 $|\rho|\le\lambda$（信号弱）：直接**压成 0**（一个"死区"）。
  When $|\rho|\le\lambda$ (weak signal): set **exactly to 0** (a "dead zone").
- 当 $|\rho|>\lambda$（信号强）：保留，但向 0 **收缩 $\lambda$**。
  When $|\rho|>\lambda$ (strong signal): keep it, but **shrink toward 0 by $\lambda$**.

这个"死区"正是 Lasso 把弱特征清零、产生稀疏的算子层面机制。（对比 Ridge：它的更新是按比例收缩，永远到不了精确的 0。）
This "dead zone" is exactly how Lasso zeros out weak features and creates sparsity at the operator level. (Ridge shrinks proportionally and never reaches exact 0.)


In [ ]:
def soft_threshold(rho, lam):
    # 软阈值: |ρ|≤λ → 0; 否则把 ρ 朝 0 拉近 λ / soft-threshold operator
    return np.sign(rho) * max(abs(rho) - lam, 0.0)

rhos = np.linspace(-3, 3, 200); lam = 1.0
out = [soft_threshold(r, lam) for r in rhos]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(rhos, out, lw=2, label=f"软阈值 soft-threshold, λ={lam}")
ax.plot(rhos, rhos, "k--", alpha=0.4, label="无惩罚 (y=ρ)")
ax.axvspan(-lam, lam, alpha=0.15, color="red", label="|ρ|≤λ → 清零 zeroed")
ax.axhline(0, color="gray", lw=0.5); ax.legend(); ax.set_xlabel("ρ"); ax.set_ylabel("w")
ax.set_title("软阈值: |ρ|≤λ 的死区被压成 0 → 产生稀疏")
plt.tight_layout(); plt.show()
print("红区(|ρ|≤λ): 系数直接=0; 红区外: 系数向0收缩 λ → Lasso 稀疏的算子机制")


<a id="4"></a>
## 4. Lasso 路径：逐个归 0 ⭐ / The Lasso Path

画系数随 λ 变化的路径，能看到 Lasso 的标志性行为：**随 λ 增大，系数一个接一个地精确归 0**——而且**噪声特征最先死**（它们和目标无关，一惩罚就撑不住）。对比 Ridge 的"平滑趋 0 但不为 0"，Lasso 是"啪地变成 0"。
Plotting coefficients vs λ shows Lasso's signature: **as λ grows, coefficients hit exactly 0 one by one** — and **noise features die first** (unrelated to the target, they collapse under any penalty). Versus Ridge's "smoothly approach 0 but never reach it", Lasso "snaps to 0".


In [ ]:
from sklearn.linear_model import Lasso
alphas = np.logspace(-3, 0.5, 60)
# 对每个 λ 拟合 Lasso 收集系数 / Lasso coefficient path
coef_path = np.array([Lasso(alpha=a, max_iter=5000).fit(Xtr, y_tr).coef_ for a in alphas])

fig, ax = plt.subplots(figsize=(9, 5))
for i in range(8):                                  # 真实特征用彩色 / real features colored
    ax.plot(alphas, coef_path[:, i], lw=1.8, label=feat_names[i])
for i in range(8, 28):                              # 噪声特征用灰色 / noise features gray
    ax.plot(alphas, coef_path[:, i], color="gray", alpha=0.3, lw=0.8)
ax.set_xscale("log"); ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("λ (alpha)"); ax.set_ylabel("系数 coefficient")
ax.legend(fontsize=7, ncol=2, title="彩色=真实, 灰=噪声"); ax.set_title("Lasso 路径: λ↑ 系数逐个精确归0(噪声最先死)")
plt.tight_layout(); plt.show()
print("灰色噪声特征率先归0, 真实特征撑得更久 → Lasso 自动识别并剔除无用特征")


<a id="5"></a>
## 5. 特征选择 + vs Ridge ⭐ / Feature Selection & vs Ridge

`LassoCV` 自动选 λ。用它在"8 真实 + 20 噪声"数据上看它剔除了多少噪声，并和 Ridge 对照（Ridge 一个都不会归零）。
`LassoCV` picks λ automatically. On the "8 real + 20 noise" data, see how much noise it removes, contrasted with Ridge (which zeros nothing).

**Lasso vs Ridge 怎么选**（面试常问）：
**Lasso vs Ridge** (often asked):
- 特征多、怀疑大部分没用、想要**稀疏可解释**模型 → **Lasso**。
  Many features, most suspected useless, want a **sparse interpretable** model → **Lasso**.
- 特征都有用、或高度相关 → **Ridge**（Lasso 在共线特征里会**随机只留一个**，不稳）。
  All features useful, or highly correlated → **Ridge** (Lasso arbitrarily keeps just one of a collinear group).
- 两者都想要 → **Elastic Net**（下一课 4.6）。
  Want both → **Elastic Net** (next, 4.6).


In [ ]:
from sklearn.linear_model import LassoCV, Ridge
lassocv = LassoCV(cv=5, max_iter=10000, random_state=0).fit(Xtr, y_tr)   # 自动选 λ
n_zero = (np.abs(lassocv.coef_) < 1e-8).sum()
n_zero_noise = (np.abs(lassocv.coef_[8:]) < 1e-8).sum()       # 被归零的噪声特征数
print(f"LassoCV 选出 λ = {lassocv.alpha_:.4f}")
print(f"系数为 0 的特征: {n_zero}/{X_aug.shape[1]} (其中 {n_zero_noise}/20 个噪声被剔除)")
print(f"存活的真实特征: {(np.abs(lassocv.coef_[:8]) > 1e-8).sum()}/8\n")

ridge = Ridge(alpha=1.0).fit(Xtr, y_tr)
print(f"对照 Ridge: 系数为 0 的特征 = {(np.abs(ridge.coef_) < 1e-8).sum()} (一个都不归零)")
print(f"\ntest R²: Lasso={lassocv.score(Xte,y_te):.4f}, Ridge={ridge.score(Xte,y_te):.4f}")
print("Lasso 自动剔除噪声 → 更简洁可解释; 噪声/无用特征多时 Lasso 常胜")


<a id="6"></a>
## 6. 小结 / Summary

```
Lasso = OLS + L1 惩罚 λΣ|wⱼ|; 把部分系数精确压到 0 → 自动特征选择
为什么稀疏(几何): L1 约束是菱形, 尖角在坐标轴上, 椭圆易先碰角 → 某坐标=0
算子机制: 软阈值 S_λ(ρ), |ρ|≤λ 的死区清零, 否则收缩 λ
Lasso 路径: λ↑ 系数逐个精确归0, 噪声特征最先死(对比 Ridge 趋0不为0)
必须标准化(同 Ridge); λ 用 LassoCV 选
选择: 想稀疏/可解释→Lasso; 特征都有用/共线→Ridge; 都要→Elastic Net(4.6)
```

### 💡 面试速查 / Interview cheat-sheet
1. **L1(Lasso) 稀疏、L2(Ridge) 只收缩**；L1 做特征选择。
   L1 (Lasso) is sparse, L2 (Ridge) only shrinks; L1 does feature selection.
2. **稀疏的几何原因**: L1 菱形尖角在坐标轴上 → 解落在轴上(某系数=0)。
   Sparsity geometry: L1's diamond corners on the axes → solution on an axis (a coefficient = 0).
3. **软阈值死区**: |ρ|≤λ 直接清零(算子机制)。
   Soft-thresholding dead zone: |ρ|≤λ → exactly 0.
4. **共线特征里 Lasso 随机只留一个** → 共线时 Ridge 更稳。
   Among collinear features Lasso keeps one arbitrarily → Ridge is steadier when collinear.
5. **必须标准化**; λ 用 CV 选。
   Must standardize; choose λ by CV.

### 下一节 / Next
**4.6 弹性网(Elastic Net)**——把 L1 和 L2 混合，兼得 Lasso 的稀疏和 Ridge 的稳定，是高维相关特征下的常用折中。
**4.6 Elastic Net** — mix L1 and L2 to get Lasso's sparsity and Ridge's stability; a common compromise for high-dimensional correlated features.
